# Lensless Computational Imaging

Reconstructs images from lensless measurements with the trained models.

`DATASET_URL` must point to a Google Drive `.zip` with this structure (matching `ImageID` filenames):
```
lensless/  ImageID.png
masks/     ImageID.npy
lensed/    ImageID.png
```

## 1. Clone the repository and install dependencies

You might need to restart the session after the requirements are installed

In [ ]:
!git clone https://github.com/aleksanderkarpov/Lensless-Computational-Imaging.git repo
%cd repo
!pip install -r requirements.txt

## 2. Set `DATASET_URL`

In [ ]:
DATASET_URL = ""

# one of: admm | unrolled | modular_pre | modular_post | modular_prepost
MODEL = "modular_prepost"

# output folder name for reconstructions
SAVE_PATH = "demo_recon"

## 3. Download model checkpoints into `checkpoints/`

In [ ]:
!python download_checkpoints.py

## 4. Download and extract the dataset

In [ ]:
import os
import zipfile

import gdown

gdown.download(DATASET_URL, "demo_data.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("demo_data.zip") as z:
    z.extractall("demo_data")

DATA_DIR = "demo_data"
for root, dirs, files in os.walk("demo_data"):
    if "lensless" in dirs:
        DATA_DIR = root
        break
print("dataset dir:", DATA_DIR)

## 5. Run inference

In [ ]:
# (model config, normalize flag, checkpoint path)
CONFIG = {
    "admm": ("admm", "true", "null"),
    "unrolled": ("unrolled", "true", "checkpoints/unrolled.pth"),
    "modular_pre": ("modular_pre", "true", "checkpoints/modular_pre.pth"),
    "modular_post": ("modular_post", "false", "checkpoints/modular_post.pth"),
    "modular_prepost": ("modular_prepost", "false", "checkpoints/modular_prepost.pth"),
}
model_config, normalize, checkpoint = CONFIG[MODEL]

In [ ]:
import subprocess

cmd = [
    "python", "inference.py",
    f"model={model_config}",
    f"normalize={str(normalize).lower()}",
    "datasets=custom_dir",
    f"datasets.test.data_dir={DATA_DIR}",
    f"inferencer.from_pretrained={checkpoint}",
    f"inferencer.save_path={SAVE_PATH}",
]

print("Running:")
print(" ".join(cmd))

subprocess.run(cmd, check=True)

In [ ]:
RECON_DIR = f"data/saved/{SAVE_PATH}/test"

## 6. Visualize (lensed (if available) vs lensless measurement vs reconstruction)

In [ ]:
import glob

import cv2
import numpy as np
import matplotlib.pyplot as plt

lensless_dir = os.path.join(DATA_DIR, "lensless")
lensed_dir = os.path.join(DATA_DIR, "lensed")
has_lensed = os.path.isdir(lensed_dir)


def load_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def find(folder, stem):
    matches = glob.glob(os.path.join(folder, stem + ".*"))
    return matches[0] if matches else None


recon_files = sorted(os.listdir(RECON_DIR))[:4]
titles = ["lensed", "lensless", "reconstruction"] if has_lensed else ["lensless", "reconstruction"]
fig, axes = plt.subplots(len(recon_files), len(titles), figsize=(4 * len(titles), 4 * len(recon_files)))
axes = np.array(axes).reshape(len(recon_files), len(titles))

for r, fname in enumerate(recon_files):
    stem = os.path.splitext(fname)[0]
    panels = []
    if has_lensed:
        lensed = find(lensed_dir, stem)
        panels.append(load_rgb(lensed) if lensed else np.zeros((8, 8, 3), np.uint8))
    ll = find(lensless_dir, stem)
    panels.append(load_rgb(ll) if ll else np.zeros((8, 8, 3), np.uint8))
    panels.append(load_rgb(os.path.join(RECON_DIR, fname)))
    for c in range(len(titles)):
        axes[r, c].imshow(panels[c])
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(titles[c])
plt.tight_layout()
plt.show()

## 7. Metrics (if `lensed/` is present)

In [ ]:
if has_lensed:
    !python calculate_metrics.py --lensed_dir $lensed_dir --recon_dir $RECON_DIR 2>/dev/null
else:
    print("No lensed provided — metrics skipped.")